In [ ]:
import os
import re
import time
import torch
import pandas as pd
import langdetect
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---------------------------------------------------------
# 1. Hardware & Model Setup
# ---------------------------------------------------------
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "GPU not detected! Enable GPU in Colab."

if 'tokenizer' not in globals():
    print(f"Loading Tokenizer: {MODEL_ID}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

if 'model' not in globals():
    print(f"Loading Model: {MODEL_ID} on {DEVICE}...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, 
        torch_dtype=torch.float16, 
        device_map="auto"
    )

# ---------------------------------------------------------
# 2. Indic Languages Configuration (Skipping 'hi')
# ---------------------------------------------------------
INDIC_LANGS = {
    "bn": "Bengali",
    "gu": "Gujarati",
    "kn": "Kannada",
    "ml": "Malayalam",
    "mr": "Marathi",
    "or": "Odia",
    "pa": "Punjabi",
    "ta": "Tamil",
    "te": "Telugu"
}

TARGET_BUDGETS = [50, 100, 200, 400, 800]
MAX_QUESTIONS = 200

# ---------------------------------------------------------
# 3. Helper Functions
# ---------------------------------------------------------
def extract_ground_truth(solution_str):
    match = re.search(r"####\s*(-?[\d,]+(?:\.\d+)?)", solution_str)
    return match.group(1).replace(",", "").strip() if match else ""

def parse_numeric_answer(text):
    if not text: return ""
    cleaned = text.replace("$", "").replace(",", "")
    numbers = re.findall(r"-?\d+(?:\.\d+)?", cleaned)
    return numbers[0].strip() if numbers else ""

def calculate_lang_ratio(text, target_code):
    sentences = [s.strip() for s in re.split(r'[.!?।\n]+', text) if s.strip()]
    if not sentences: return 0.0
    target_count = 0
    for sent in sentences:
        try:
            if langdetect.detect(sent) == target_code: 
                target_count += 1
        except: 
            pass
    return (target_count / len(sentences)) * 100

In [3]:
ANSWER_TRIGGERS = {
    "en": "Final Answer:",
    "hi": "अंतिम उत्तर:",
    "bn": "চূড়ান্ত উত্তর:",
    "gu": "અંતિમ જવાબ:",
    "kn": "ಅಂತಿಮ ಉತ್ತರ:",
    "ml": "അന്തിമ ഉത്തരം:",
    "mr": "अंतिम उत्तर:",
    "or": "ଚୂଡ଼ାନ୍ତ ଉତ୍ତର:",
    "pa": "ਅੰਤਿਮ ਜਵਾਬ:",
    "ta": "இறுதிப் பதில்:",
    "te": "తుది సమాధానం:"
}

In [ ]:
print(f"Starting Multi-Language Loop for {len(INDIC_LANGS)} languages (Max {MAX_QUESTIONS} questions each)...\n")
start_time_global = time.time()

for lang_code, lang_name in INDIC_LANGS.items():
    output_csv = f"gsm8k_{lang_code}_results.csv"
    
    print("=" * 60)
    print(f"Loading {lang_name} ({lang_code}) dataset...")
    
    raw_dataset = load_dataset("sarvamai/gsm8k-indic", lang_code, split="test")
    subset_size = min(MAX_QUESTIONS, len(raw_dataset))
    eval_dataset = raw_dataset.select(range(subset_size))
    
    completed_sample_ids = set()
    if os.path.exists(output_csv):
        existing_df = pd.read_csv(output_csv)
        if "sample_id" in existing_df.columns:
            completed_sample_ids = set(existing_df["sample_id"].astype(str))
        print(f"🔄 Resuming {lang_name} run. Found {len(completed_sample_ids)} finished evaluations.")
    
    print(f"⚡ Processing {subset_size} questions across {len(TARGET_BUDGETS)} budgets for {lang_name}...")
    
    for idx, item in enumerate(eval_dataset):
        question_id = f"GSM8K_{lang_code.upper()}_{idx+1}"
        question_text = item["question"]
        ground_truth = extract_ground_truth(item["answer"])
        
        messages = [{"role": "user", "content": question_text}]
        base_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(base_prompt, return_tensors="pt").to(DEVICE)
        prompt_len = inputs["input_ids"].shape[1]
        
        for budget in TARGET_BUDGETS:
            sample_id = f"{question_id}_b{budget}"
            
            if sample_id in completed_sample_ids:
                continue
                
            # --- STEP 1: Generate Thinking ---
            t0 = time.time()
            gen_output = model.generate(
                **inputs,
                max_new_tokens=budget,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
            torch.cuda.synchronize()
            step1_time = time.time() - t0
            
            raw_gen_text = tokenizer.decode(gen_output[0][prompt_len:], skip_special_tokens=False)
            
            if "</think>" in raw_gen_text:
                thinking_trace = raw_gen_text.split("</think>")[0].replace("<think>", "").strip()
                cutoff_status = "Finished_Thinking_Naturally"
            else:
                thinking_trace = raw_gen_text.replace("<think>", "").strip()
                cutoff_status = "Thinking_Cut_Off_By_Budget"
                
            thinking_tokens_used = len(tokenizer.encode(thinking_trace))
            lang_ratio = calculate_lang_ratio(thinking_trace, lang_code)
            
            # --- STEP 2: Force Immediate Answer ---
            trigger_text = ANSWER_TRIGGERS.get(lang_code, "Final Answer:")
            continuation_prompt = f"{base_prompt}<think>\n{thinking_trace}\n</think>\n\n{trigger_text} "
            forced_inputs = tokenizer(continuation_prompt, return_tensors="pt").to(DEVICE)
            forced_prompt_len = forced_inputs["input_ids"].shape[1]
            
            t0 = time.time()
            ans_output = model.generate(
                **forced_inputs,
                max_new_tokens=15,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
            torch.cuda.synchronize()
            step2_time = time.time() - t0
            
            raw_ans_text = tokenizer.decode(ans_output[0][forced_prompt_len:], skip_special_tokens=True).strip()
            parsed_answer = parse_numeric_answer(raw_ans_text)
            is_correct = (parsed_answer == ground_truth)
            
            # --- STEP 3: Save Detailed Record ---
            record = {
                "sample_id": sample_id,
                "question_id": question_id,
                "language": lang_name,
                "model": MODEL_ID,                           
                "question": question_text,                   
                "max_cutoff_limit": budget,                 
                "tokens_used": thinking_tokens_used,
                "full_thinking_text": thinking_trace,
                "raw_answer_text": raw_ans_text,
                "parsed_answer": parsed_answer,
                "ground_truth": ground_truth,
                "correct": is_correct,
                "step1_time_sec": round(step1_time, 2),
                "step2_time_sec": round(step2_time, 2),
                "target_lang_ratio_pct": round(lang_ratio, 1),
                "cutoff_status": cutoff_status
            }
            
            df_row = pd.DataFrame([record])
            df_row.to_csv(output_csv, mode="a", index=False, header=not os.path.exists(output_csv))
            completed_sample_ids.add(sample_id)


        # Print progress every 10 questions
        if (idx + 1) % 10 == 0 or (idx + 1) == subset_size:
            elapsed = time.time() - start_time_global
            print(f"[{lang_name}] Progress: {idx + 1}/{subset_size} | Global Elapsed: {elapsed/60:.1f} mins")

    print(f"Finished {lang_name}. Results saved to `{output_csv}`\n")

total_mins = (time.time() - start_time_global) / 60
print("=" * 60)
print(f"ALL MULTI-LANGUAGE TRIALS COMPLETE IN {total_mins:.1f} MINS!")